# Simple Chat Notebook (Transformers)
This notebook loads a Hugging Face causal LM checkpoint and provides a tiny chat UI.

**Model repo**: `gcyzsl/O3_LLAMA2_ScienceQA`

If the repo is gated/private, set `HF_TOKEN` in your environment or run `huggingface-cli login` first.

In [1]:
# (Optional) If needed, install dependencies
# !pip -q install -U "transformers>=4.41.0" accelerate sentencepiece ipywidgets

import os, torch
from transformers import AutoTokenizer, AutoModelForCausalLM


In [2]:
MODEL_ID = "gcyzsl/O3_LLAMA2_ScienceQA"

# If you need an auth token, set one of these:
# os.environ["HF_TOKEN"] = "YOUR_TOKEN"
# os.environ["HUGGINGFACE_HUB_TOKEN"] = "YOUR_TOKEN"

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if (device == "cuda" and torch.cuda.is_bf16_supported()) else (torch.float16 if device == "cuda" else torch.float32)

print("device:", device, "dtype:", dtype)


device: cuda dtype: torch.bfloat16


In [3]:
# Load tokenizer + model
tok = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto" if device == "cuda" else None,
)

# Some LLaMA-style tokenizers don't have pad_token by default
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model.eval()
print("Loaded:", MODEL_ID)


/dataset/dataset/envs/hai_ssh_o3/lib/python3.10/site-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/dataset/dataset/envs/hai_ssh_o3/lib/python3.10/site-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loaded: gcyzsl/O3_LLAMA2_ScienceQA


In [4]:
# Generation helper
@torch.inference_mode()
def generate_reply(messages, max_new_tokens=256, temperature=0.7, top_p=0.9, do_sample=True):
    """messages: list of dicts like [{"role":"user","content":"..."}]"""
    # Prefer chat template when available
    if hasattr(tok, "apply_chat_template"):
        input_ids = tok.apply_chat_template(
            messages,
            add_generation_prompt=True,
            return_tensors="pt",
        )
    else:
        # Fallback: simple concatenation
        text = ""
        for m in messages:
            role = m["role"]
            content = m["content"]
            text += f"{role.upper()}: {content}\n"
        text += "ASSISTANT: "
        input_ids = tok(text, return_tensors="pt").input_ids

    input_ids = input_ids.to(model.device)

    gen = model.generate(
        input_ids=input_ids,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=temperature if do_sample else None,
        top_p=top_p if do_sample else None,
        pad_token_id=tok.pad_token_id,
        eos_token_id=tok.eos_token_id,
    )

    # Decode full then strip prompt part
    full = tok.decode(gen[0], skip_special_tokens=True)

    if hasattr(tok, "apply_chat_template"):
        prompt = tok.decode(input_ids[0], skip_special_tokens=True)
        if full.startswith(prompt):
            return full[len(prompt):].strip()
    else:
        # Try to split on last "ASSISTANT:"
        if "ASSISTANT:" in full:
            return full.split("ASSISTANT:")[-1].strip()

    return full.strip()


## Tiny Chat UI
Type a message and click **Send**. Conversation is kept in-memory (list of messages).

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

chat_history = []  # list of {"role": "...", "content": "..."}

out = widgets.Output(layout={"border": "1px solid #ddd", "padding": "10px", "height": "320px", "overflow_y": "auto"})
inp = widgets.Textarea(
    placeholder="Type your message...",
    layout=widgets.Layout(width="100%", height="90px"),
)
send_btn = widgets.Button(description="Send", button_style="primary")
clear_btn = widgets.Button(description="Clear", button_style="")
max_new = widgets.IntSlider(value=256, min=16, max=1024, step=16, description="max_new")
temp = widgets.FloatSlider(value=0.7, min=0.0, max=1.5, step=0.05, description="temp")
top_p = widgets.FloatSlider(value=0.9, min=0.1, max=1.0, step=0.05, description="top_p")
sample = widgets.Checkbox(value=True, description="do_sample")

def render():
    with out:
        clear_output(wait=True)
        for m in chat_history:
            role = m["role"]
            content = m["content"]
            if role == "user":
                display(Markdown(f"**You:** {content}"))
            else:
                display(Markdown(f"**Assistant:** {content}"))

def on_send(_):
    text = inp.value.strip()
    if not text:
        return
    inp.value = ""
    chat_history.append({"role": "user", "content": text})

    # Generate
    reply = generate_reply(
        chat_history,
        max_new_tokens=int(max_new.value),
        temperature=float(temp.value),
        top_p=float(top_p.value),
        do_sample=bool(sample.value),
    )
    chat_history.append({"role": "assistant", "content": reply})
    render()

def on_clear(_):
    chat_history.clear()
    render()

send_btn.on_click(on_send)
clear_btn.on_click(on_clear)

ui = widgets.VBox([
    out,
    inp,
    widgets.HBox([send_btn, clear_btn]),
    widgets.HBox([max_new, temp]),
    widgets.HBox([top_p, sample]),
])

render()
display(ui)


: 